# Qwen


In [276]:
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
import torch
import time

In [19]:
model = 'Qwen/Qwen2.5-0.5B'
tokenizer = AutoTokenizer.from_pretrained(model)
qwen = AutoModel.from_pretrained(model)
causal_lm = AutoModelForCausalLM.from_pretrained(model)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1742.64it/s]
c:\Users\skd92\.conda\envs\Dragoon\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\skd92\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [ ]:
def generate_response(prompt, strategy, model, tokenizer, n=100, T=1, k=None, p=None, eos_id=None):
    """
    Enter a prompt to generate the next tokens in a loop until maximum token limit or end of sequence or user interruption occurs.\n
    model = GPT decoder model\n
    tokenizer = tokenizer same as the model, mixing different tokenizer might result in broken text generation\n
    strategy = ["Greedy", "Temperature", "Top-k", "Top-p"]\n
    n = number of tokens, T = temperature, k = k in top-k, p = p in top-p\n
    eos_id -> end of sequence id used to stop the text generation on reaching end of sequence. If set None the model keeps on generating text until it reaches the maximum token limit.\n
    """

    temperature=T
    eos = eos_id
    past_key_values=None
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids

    if strategy=='Greedy':
        for x in range(n):
            with torch.no_grad():
                base_output = model(input_ids=input_ids, past_key_values=past_key_values, use_cache=True)
                hidden_states = base_output.last_hidden_state
                base_logits = causal_lm.lm_head(hidden_states)
                past_key_values=base_output.past_key_values

            scaled_logits = base_logits[:, -1, :]/temperature
            probs = torch.softmax(scaled_logits, dim=-1)

            next_token_id = torch.argmax(probs ,dim=-1, keepdim=True)
            input_ids = next_token_id

            if next_token_id==eos:
                break

            new_text = tokenizer.decode(next_token_id[0])
            print(new_text, end="", flush=True)
        
    elif strategy=="Temperature":
        for x in range(n):
            with torch.no_grad():
                base_output = model(input_ids=input_ids)
                hidden_state = base_output.last_hidden_state
                base_logits = causal_lm.lm_head(hidden_state)

            scaled_logits = base_logits[:, -1, :]/temperature
            probs = torch.softmax(scaled_logits, dim=-1)

            next_token_id = torch.multinomial(probs, num_samples=1)

            if next_token_id==eos:
                break

            input_ids = torch.cat([input_ids, next_token_id], dim=-1)

            new_text = tokenizer.decode(next_token_id[0])
            print(new_text, end="", flush=True)

    elif strategy=="Top-k":
        for x in range(n):
            with torch.no_grad():
                bo = model(input_ids=input_ids)
                hs = bo.last_hidden_state
                base_logits = causal_lm.lm_head(hs)

            logits = base_logits[:, -1, :]

            scaled_logits = logits/temperature

            values, indices = torch.topk(scaled_logits, k)
            probs = torch.softmax(values, dim=-1)

            sample = torch.multinomial(probs, 1)
            next_token_id = indices.gather(-1, sample)
            input_ids = torch.cat([input_ids, next_token_id],  dim=-1)

            if next_token_id==eos:
                break

            new_text = tokenizer.decode(next_token_id[0])
            print(new_text, end="", flush=True)
    
    elif strategy=="Top-p":
        for x in range(n):
            with torch.no_grad():
                base_output = model(input_ids=input_ids)
                hs = base_output.last_hidden_state
                base_logits = causal_lm.lm_head(hs)

            logits = base_logits[:, -1, :]
            scaled_logits = logits/temperature
            sorted_logits, sorted_indices = torch.sort(scaled_logits, descending=True)

            sorted_probs = torch.softmax(sorted_logits, dim=-1)
            cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
            mask = cumulative_probs<=p
            mask[..., 0] = True

            filtered_probs = sorted_probs * mask
            filtered_probs /= filtered_probs.sum(dim=-1)

            sample = torch.multinomial(filtered_probs, num_samples=1)

            next_token_id = sorted_indices.gather(-1, sample)
            input_ids = torch.cat([input_ids, next_token_id], dim=-1)
            if next_token_id == eos:
                break

            new_text = tokenizer.decode(next_token_id[0])
            print(new_text, end="", flush=True)
    
    else:
        print("Invalid strategy. Enter a valid strategy to generate tokens.")

In [306]:
output_lengths = [20, 100, 300, 500]

record_greedy_time = {}
for x in output_lengths:
    start = time.perf_counter()
    generate_response(
        prompt="Explain about Harry Potter and the Philosopher's Stone.",
        strategy="Greedy",
        model=qwen,
        tokenizer=tokenizer,
        n=x,
        eos_id=tokenizer.eos_token_id
    )
    latency_seconds = time.perf_counter() - start
    if x not in record_greedy_time.keys():
        record_greedy_time[x] = []
    record_greedy_time[x] = latency_seconds

TypeError: argument 'ids': Can't extract `str` to `Vec`

In [ ]:

device

device(type='cpu')